# 0623ver_2 老師逐字稿十點任務回應  
## 具體檔案改動舉例版

版本標準：`0623ver_2_最終整合版_CP950_Excel直開修正版`  
報告日期：`2026-06-23`  
相對路徑基準：所有檔案路徑皆以 `0623ver_2/` 為根目錄。

---

## 這份報告的修正重點

前一版只寫「整理 API 橋接」、「保留 runtime service」這種說法，確實太空泛。  
這份改成逐項說明：

```text
1. 原本要解決什麼問題
2. 新增了什麼檔案或資料夾
3. 修改了什麼檔案
4. 檔案裡具體改了什麼內容
5. 舉例：程式或 CSV / SQL 裡實際出現的內容
6. 這樣如何達成老師要求
7. 要怎麼驗證
```

---

## 0623ver_2 的版本定位

```text
0623ver_2
= final_version 可實際啟動的 API / UI / DB 主系統
+ 0620 hotfix
+ 0617_B 少榆端核心 CSV / knowledge / schema / service reference / future contract
+ Ontology / TTL / Rule 推論
+ Sequence Diagram
+ CSV CP950 / Excel 直開修正
```


# 0. Runtime 與 Reference 的分工

## 主旨

最後版本不能只會展示檔案，也要能實際跑 API；但也不能把 0617_B 你做過的資料都丟掉。  
所以 `0623ver_2` 分成兩層：

## Runtime：實際會被 Docker / API / UI 使用

```text
api/
database/
dataprocess/
rules/
services/
ui/
config/knowledge/
ontology/
start.ps1
docker-compose.yml
```

## Reference：保留 0617_B 少榆端核心資料，供報告與追溯

```text
csv_templates/
knowledge/
schema/
templates/
docs/0617_B_reference/
service_reference_0617_B/
database_versionB_reference_0617_B/
ontology/0617_B_modular_reference/
```

## 具體處理方式

不是把過去所有 CSV 全部塞進來，而是只保留老師會問到、少榆端會用到、或報告需要追溯的核心資料。  
例如：

| 類型 | 放置位置 | 作用 |
|---|---|---|
| 實際 API | `api/` | FastAPI 端點與 UI / service bridge |
| 實際 DB schema | `database/setup_db.sql` | 建立 PostgreSQL table 與 seed |
| 實際 service | `services/` | Future / Integrated / Monitoring / Troubleshooting runtime |
| 0617_B service 原始碼 | `service_reference_0617_B/` | 保留你之前做的 service source，供追溯 |
| 0617_B CSV | `csv_templates/`、`knowledge/`、`docs/0617_B_reference/contracts/` | 保留 sensor / alert / future / knowledge 對應 |
| 目前 Ontology Rule | `ontology/threshold_reference.csv`、`ontology/sprayline_threshold.ttl`、`ontology/rule_inference.py` | 可測 CSV → TTL → Rule |


# 1. 把 UI → API → Service → DB → UI 的流程串起來

## 主旨

原本老師看不出 UI 是怎麼拿到資料的，也看不出 API、Service、DB 之間的實際呼叫關係。這項任務不是只說有串接，而是要讓 UI 能透過 API 拿到 PostgreSQL 的資料，再回到畫面。

## 新增了什麼檔案或資料夾

| 新增檔案 / 資料夾 | 用途 |
|---|---|
| `無新增單一獨立檔案` | 此項主要是把既有 API、Bridge、DB、UI service 串起來。 |

## 修改了什麼檔案內容

| 修改檔案 / 資料夾 | 具體修改內容 |
|---|---|
| `api/api_server.py` | 新增 / 整理 UI 用 endpoint，例如 `/api/time-series/ui/summary`、`/api/time-series/ui/station-detail`、`/api/time-series/ui/component-detail`，並在 endpoint 中呼叫 `_run_integrated_for_ui()`。 |
| `api/shaoyu_ui_bridge.py` | 加入 `COMPONENT_METRICS`、`COMPONENT_ALIASES`、`build_ui_summary_output()`、`build_ui_station_detail_output()`、`build_ui_component_detail_output()`，把 integrated service 結果轉成 UI JSON。 |
| `api/time_series_service.py` | 整理 time-series 查詢處理，讓 Past / Current / Future 資料可被統一包成 viewer state / summary。 |
| `ui/engineer/app/services/api_data_service.py` | 前端 API client 改成呼叫後端 station-detail / component-detail，並把回傳資料轉成前端可畫圖的 snapshot / trend points。 |
| `database/setup_db.sql` | 確認 DB 有 sensor_1min、sensor_3min、future_prediction_result、alert_event、catalog 等 API 會查的表。 |

## 舉例：檔案裡實際改了什麼

| 檔案 | 具體例子 |
|---|---|
| ``api/api_server.py`` | 在 `/api/time-series/ui/summary` 裡，如果 UI 沒帶 `slider_value`，預設補 `0`，再呼叫 `_run_integrated_for_ui()`；若來源是 integrated，就回 `build_ui_summary_output(data)`，若 integrated 失敗則 fallback demo。 |
| ``api/api_server.py`` | 在 `/api/time-series/ui/station-detail` 裡檢查 `line_id` 或 `station_id`，再把 `line_scope` 補進 request，確保 UI 傳 M1 / line_1 時 API 能轉成 Station_1。 |
| ``api/shaoyu_ui_bridge.py`` | 新增 `COMPONENT_ALIASES`，讓 `QUALITY`、`quality`、`quality_module` 都會被轉成 `quality_module`；`AIR_COMPRESSOR`、`air`、`air_compressor` 都會被轉成 `air_compressor`。 |
| ``ui/engineer/app/services/api_data_service.py`` | 把後端回傳的 `current_snapshot` 轉成前端 point，明確抓 `film_thickness_um`、`air_pressure_bar`、`spray_width_mm`、`filter_diff_pressure_bar` 等欄位。 |

## 範例片段

```python
# api/api_server.py 舉例
@app.post("/api/time-series/ui/summary")
def HandleTimeSeriesUiSummaryRequest(request):
    if "slider_value" not in request:
        request["slider_value"] = 0
    result = _run_integrated_for_ui(request, fallback_to_demo=True)
    if result.get("source_mode") == "integrated":
        return JSONResponse(content=build_ui_summary_output(result.get("data", {})))
```

## 這樣如何達成任務

這一項不是只保留檔案，而是在 API 端新增 UI 專用 endpoint，Bridge 端補 component mapping，UI 端改成讀 API 回傳的 snapshot / trend points，DB 端補齊 sensor 與 catalog 表，形成 UI → API → Service → DB → UI 的完整資料流。

## 驗證方式

```powershell
cd "C:\Users\jed92\Desktop\Project-SprayLine\0623ver_2"
powershell -ExecutionPolicy Bypass -File .\start.ps1 -Mode start -WithData
docker compose ps
```

打開：

```text
http://localhost:8011/docs
http://localhost:8013
```

確認 Engineer UI 有站點資料、零件狀態、趨勢資料。


# 2. 補 Sequence Diagram，說明 UI 呼叫誰、Service 呼叫誰、最後誰回傳

## 主旨

老師問的是呼叫順序，不是單純要一張架構圖。因此需要畫出 UI、API、Service、DB、Rule、TTL 之間誰先呼叫誰。

## 新增了什麼檔案或資料夾

| 新增檔案 / 資料夾 | 用途 |
|---|---|
| `docs/sequence_diagram.md` | 新增 Mermaid 時序圖，分成三段流程：UI 查詢線、Service Orchestration 線、Ontology Rule 線。 |

## 修改了什麼檔案內容

| 修改檔案 / 資料夾 | 具體修改內容 |
|---|---|
| `docs/sequence_diagram.md` | 將老師關心的參與者明確寫入 Mermaid：Engineer UI、FastAPI API Server、UI Bridge、Integrated Service、PostgreSQL Database、Ontology Rule Engine、TTL Threshold Knowledge。 |

## 舉例：檔案裡實際改了什麼

| 檔案 | 具體例子 |
|---|---|
| `第一張圖` | UI 呼叫 API：`GET dashboard data`，API 再呼叫 Bridge 做 station / component name normalize。 |
| `第一張圖` | Service 查 DB：`Service -> DB: SELECT sensor_1min and sensor_3min`。 |
| `第一張圖` | Rule 讀 TTL：`Service -> Rule: infer_state metric and value`，再 `Rule -> TTL: Read threshold TTL`。 |
| `第二張圖` | Integrated service 做 future、monitoring、alert、DB write-back，包含 `INSERT future_prediction_result`、`INSERT alert_event and alert links`、`UPDATE batch_station_status`。 |

## 範例片段

```mermaid
UI->>API: GET dashboard data
API->>Bridge: Normalize station and component names
Bridge->>Service: Query station summary and component detail
Service->>DB: SELECT sensor_1min and sensor_3min
Service->>Rule: infer_state metric and value
Rule->>TTL: Read threshold TTL
API-->>UI: Render station status component status and trend chart
```

## 這樣如何達成任務

我不是只畫一張總架構，而是把老師問的呼叫順序畫成三條時序圖：UI 查詢線、service orchestration 線、CSV→TTL→Rule 推論線。

## 驗證方式

打開：

```text
docs/sequence_diagram.md
```

用 VS Code Mermaid Preview 或 Mermaid Live 確認三張圖可顯示，並截圖放報告。


# 3. 整合 Past / Current / Future 查詢 API

## 主旨

老師要知道時間軸 Past / Current / Future 怎麼查，不希望每個時間點都是分散測試或假資料。

## 新增了什麼檔案或資料夾

| 新增檔案 / 資料夾 | 用途 |
|---|---|
| `無新增單一獨立檔案` | 此項是把既有 API、time-series service、Bridge、UI data service 整理成同一條查詢線。 |

## 修改了什麼檔案內容

| 修改檔案 / 資料夾 | 具體修改內容 |
|---|---|
| `api/api_server.py` | 保留 `/api/time-series/demo/current`、`/api/time-series/demo/past`、`/api/time-series/demo/future`，並新增 UI summary / detail endpoint。 |
| `api/service_orchestration_adapter.py` | 在 `run_integrated_service_demo()` 中建立 `slider_map = {'past': -60, 'current': 0, 'future': 30}`，將 past/current/future 轉成 slider_value。 |
| `api/time_series_service.py` | 整理 `DetermineTimeType()`、`QueryRawDataFromDatabase()`、`BuildViewerState()`、`GetSampleMethod()` 等流程。 |
| `api/shaoyu_ui_bridge.py` | 把 UI 傳入的 `line_id`、`line_scope`、`station_scope` 轉換成 integrated service 能接受的 `station_scope`。 |

## 舉例：檔案裡實際改了什麼

| 檔案 | 具體例子 |
|---|---|
| ``api/service_orchestration_adapter.py`` | 用 `slider_map` 把 `past` 對應 `-60`，`current` 對應 `0`，`future` 對應 `30`，再建立 integrated request。 |
| ``api/shaoyu_ui_bridge.py`` | `build_integrated_request_from_ui()` 會從 `station_scope`、`line_scope`、`line_id` 中取一個，轉成後端的 Station scope。 |
| ``api/time_series_service.py`` | 在 sample method 中納入 `film_thickness_um`、`target_film_thickness_um`、`air_pressure_bar`、`spray_width_mm` 等欄位的抽樣方式。 |

## 範例片段

```python
# api/service_orchestration_adapter.py 舉例
slider_map = {
    "past": -60,
    "current": 0,
    "future": 30,
}
request = build_integrated_request(
    slider_value=slider_map[time_type],
    station_scope=station_id,
    window_minutes=window_minutes,
)
```

## 這樣如何達成任務

Past / Current / Future 不是各自獨立，我用 slider_value 與 time-series API 整合，讓 UI 可以透過同一套 summary / detail endpoint 取得不同時間窗資料。

## 驗證方式

Swagger 打開：

```text
http://localhost:8011/docs
```

測：

```text
/api/time-series/demo/current
/api/time-series/demo/past
/api/time-series/demo/future
/api/time-series/ui/summary
/api/time-series/ui/station-detail
/api/time-series/ui/component-detail
```

Engineer UI 拖時間軸或切換資料時，畫面能取得對應資料。


# 4. 整合 Future Prediction / Monitoring / Alert / Troubleshooting / DB write-back

## 主旨

老師不只要 function，而是要看一個完整 service 流程：Future prediction、Monitoring、Alert、Troubleshooting 是否能經 API 觸發並寫回 DB。

## 新增了什麼檔案或資料夾

| 新增檔案 / 資料夾 | 用途 |
|---|---|
| `service_reference_0617_B/` | 新增整理 0617_B 原始 future_service、integrated_service、monitoring_worker、troubleshooting_service 等來源，供追溯。 |
| `database_versionB_reference_0617_B/` | 新增整理 0617_B Database/versionB 參考檔，保留 DB 對接來源。 |

## 修改了什麼檔案內容

| 修改檔案 / 資料夾 | 具體修改內容 |
|---|---|
| `api/service_orchestration_adapter.py` | 加入 `build_integrated_request()`、`run_integrated_service_query()`、`build_future_prediction_payload_for_api()`、`save_future_prediction_payload_for_api()`、`run_monitoring_once_for_api()`。 |
| `api/api_server.py` | 新增 service orchestration endpoint，例如 `/api/service-orchestration/integrated/query`、`/run-once`、`/future/save`、`/monitoring/run`、`/troubleshooting/matrix`。 |
| `database/setup_db.sql` | 補 `future_prediction_result`、`alert_event`、`alert_cause_link`、`alert_response_link`、catalog / map table。 |
| `config/knowledge/troubleshooting_matrix_reference.csv` | 補 runtime 可讀的 troubleshooting matrix reference，供 API / adapter 產生 troubleshooting payload。 |

## 舉例：檔案裡實際改了什麼

| 檔案 | 具體例子 |
|---|---|
| ``api/service_orchestration_adapter.py`` | `run_integrated_service_query()` 直接 import `run_integrated_once`，並把 `write_back` 傳進去，讓 API 可以觸發 integrated service。 |
| ``api/service_orchestration_adapter.py`` | `save_future_prediction_payload_for_api()` 會建立 future prediction payload，拿 DB connection，呼叫 `save_future_prediction_result()` 寫回資料庫。 |
| ``api/service_orchestration_adapter.py`` | `run_monitoring_once_for_api()` 會呼叫 monitoring worker 的 `run_monitoring_once()`。 |
| ``api/api_server.py`` | `RunIntegratedServiceOnce()` 會強制 `request['write_back'] = True`，代表這條線是要測 DB write-back。 |

## 範例片段

```python
# api/api_server.py 舉例
@app.post("/api/service-orchestration/integrated/run-once")
def RunIntegratedServiceOnce(request):
    request["write_back"] = True
    return RunIntegratedServiceQuery(request)
```

```python
# api/service_orchestration_adapter.py 舉例
from integrated_service.sprayline_integrated_service import run_integrated_once
result = run_integrated_once(request=request, write_back=write_back)
```

## 這樣如何達成任務

我不是只保留 service 資料夾，而是用 API service_orchestration_adapter 把 integrated service、future save、monitoring run、troubleshooting matrix 都包成 API 可呼叫流程，並且搭配 DB table 寫回。

## 驗證方式

Swagger 測試：

```text
POST /api/service-orchestration/integrated/run-once
POST /api/service-orchestration/future/save
POST /api/service-orchestration/monitoring/run
```

DBeaver 查：

```sql
SELECT * FROM future_prediction_result ORDER BY created_at DESC LIMIT 20;
SELECT * FROM alert_event ORDER BY ts DESC LIMIT 20;
```


# 5. 說清楚 threshold 來源 CSV 在哪裡

## 主旨

老師問 threshold 來源，不能回答『程式裡有』，要指出來源 CSV 與它和 runtime JSON、ontology TTL 的關係。

## 新增了什麼檔案或資料夾

| 新增檔案 / 資料夾 | 用途 |
|---|---|
| `ontology/threshold_reference.csv` | 新增 / 整理目前 CSV→TTL→Rule 的正式 threshold source。 |
| `csv_templates/sensor_threshold_reference.csv` | 保留 0617_B threshold template。 |
| `knowledge/legacy_thresholds_from_SprayLine_3/` | 保留舊版 filter / nozzle / process threshold 追溯資料。 |

## 修改了什麼檔案內容

| 修改檔案 / 資料夾 | 具體修改內容 |
|---|---|
| `ontology/threshold_reference.csv` | 改為 CP950 / Big5 相容並加入 `sep=,`，讓 Windows Excel 直接雙擊比較不亂碼且能分欄。 |
| `ontology/threshold_to_ttl.py` | 加入多編碼讀取，支援 `utf-8-sig`、`utf-8`、`cp950`、`big5`、`gb18030`，並自動略過第一行 `sep=,`。 |
| `rules/sensor_thresholds.json` | 保留 runtime API / service 使用的 JSON threshold，與 DB / ontology 中的重要門檻對齊。 |

## 舉例：檔案裡實際改了什麼

| 檔案 | 具體例子 |
|---|---|
| ``ontology/threshold_reference.csv`` | 包含 metric、component、unit、normal_min、normal_max、warning_low_min、warning_high_max、fault_low_max、fault_high_min、cause_id、response_ids 等欄位。 |
| ``ontology/threshold_to_ttl.py`` | 新增 `_read_csv_text()`，先以多種編碼讀檔，若第一行是 `sep=,` 就跳過，避免 Excel 直開格式破壞 CSV parser。 |
| ``rules/sensor_thresholds.json`` | air_pressure_bar 的 normal 改為 2.7–3.8；spray_width_mm 的 normal 改為 70–130。 |

## 範例片段

```python
# ontology/threshold_to_ttl.py 舉例
def _read_csv_text(csv_path: Path) -> str:
    data = csv_path.read_bytes()
    for enc in ("utf-8-sig", "utf-8", "cp950", "big5", "gb18030"):
        try:
            text = data.decode(enc)
            break
        except UnicodeDecodeError:
            continue

    lines = text.splitlines()
    if lines and lines[0].strip().lower().replace(" ", "") == "sep=,":
        lines = lines[1:]
    return "\n".join(lines)
```

## 這樣如何達成任務

threshold 的來源不是口頭說明，而是 `ontology/threshold_reference.csv`；它可用 Excel 直接看，也能用 `threshold_to_ttl.py` 轉成 TTL 給 rule_inference.py 使用。

## 驗證方式

直接用 Excel 打開：

```text
ontology/threshold_reference.csv
csv_templates/sensor_threshold_reference.csv
```

再用程式測：

```powershell
python ontology\threshold_to_ttl.py --csv ontology\threshold_reference.csv --ttl ontology\sprayline_threshold.ttl
```

預期：

```text
[OK] TTL generated: ontology/sprayline_threshold.ttl
```


# 6. 補 CSV → TTL → Rule 推論流程

## 主旨

老師要看資料如何從 CSV 變成可推論的 ontology / rule，而不是只有 CSV 或只有 hard code。

## 新增了什麼檔案或資料夾

| 新增檔案 / 資料夾 | 用途 |
|---|---|
| `ontology/threshold_to_ttl.py` | 新增 CSV 轉 TTL 程式。 |
| `ontology/sprayline_threshold.ttl` | 新增 / 產生 Rule 推論用 TTL knowledge file。 |
| `ontology/rule_inference.py` | 新增讀 TTL 並推論 normal / warning / fault 的 rule engine。 |
| `scripts/run_ontology_rule_test.py` | 新增一鍵執行 rule smoke test 的腳本。 |
| `docs/ontology_rule_test_result.json` | 新增 smoke test 輸出結果保存。 |

## 修改了什麼檔案內容

| 修改檔案 / 資料夾 | 具體修改內容 |
|---|---|
| `ontology/rule_inference.py` | 修正 TTL block parser：不用單純用 `.` 切斷，避免小數點 `2.7`、`13.5` 被錯切；並補 `_between()` 避免上下限皆空時誤判。 |
| `ontology/threshold_to_ttl.py` | 補多編碼與 `sep=,` 處理，確保 CP950 CSV 仍能轉 TTL。 |

## 舉例：檔案裡實際改了什麼

| 檔案 | 具體例子 |
|---|---|
| ``threshold_to_ttl.py`` | 將每列 CSV 轉成 `sl:Threshold_metric` 的 TTL individual，寫入 normal / warning / fault 門檻、component、unit、cause_id、response_ids。 |
| ``rule_inference.py`` | `load_rules()` 讀取 TTL，整理成 `ThresholdRule` dataclass。 |
| ``rule_inference.py`` | `infer_state(metric, value)` 比較 value 與 normal / warning / fault 範圍，回傳 `state`、`level`、`issue`、`cause_id`、`response_ids`。 |
| ``rule_inference.py`` | smoke test 包含 `air_pressure_bar=3.2`、`air_pressure_bar=4.1`、`spray_width_mm=145`、`film_thickness_um=17.8` 等案例。 |

## 範例片段

```python
# ontology/rule_inference.py 舉例
def infer_state(metric: str, value: float, ttl_path="ontology/sprayline_threshold.ttl"):
    rules = load_rules(ttl_path)
    rule = rules[metric]
    if _between(value, rule.normal_min, rule.normal_max):
        state = "normal"
    elif warning_condition:
        state = "warning"
    else:
        state = "fault"
    return {"metric": metric, "value": value, "state": state,
            "cause_id": rule.cause_id, "response_ids": rule.response_ids}
```

## 這樣如何達成任務

我新增一條可測試的 CSV→TTL→Rule pipeline。CSV 是來源，TTL 是知識表示，rule_inference.py 讀 TTL 後輸出 normal / warning / fault 與對應的原因、處置建議。

## 驗證方式

```powershell
python ontology\threshold_to_ttl.py --csv ontology\threshold_reference.csv --ttl ontology\sprayline_threshold.ttl
python ontology\rule_inference.py --run-smoke-test
```

預期重點：

```text
air_pressure_bar 3.2 -> normal
air_pressure_bar 4.1 -> fault
spray_width_mm 145 -> fault
film_thickness_um 17.8 -> fault
```


# 7. 補 Protégé 可展示的完整 ontology

## 主旨

程式推論用的 TTL 主要是 threshold，Protégé 看起來會太空；老師展示時需要看到產線、站點、元件、感測指標、狀態、原因、處置之間的關係。

## 新增了什麼檔案或資料夾

| 新增檔案 / 資料夾 | 用途 |
|---|---|
| `ontology/sprayline_full_ontology.ttl` | 新增 Protégé 展示用 full ontology。 |
| `ontology/0617_B_modular_reference/` | 整理 0617_B 原本 modular ontology reference。 |
| `docs/full_ontology_protege_guide.md` | 新增 Protégé 開啟與查看說明。 |

## 修改了什麼檔案內容

| 修改檔案 / 資料夾 | 具體修改內容 |
|---|---|
| `ontology/sprayline_full_ontology.ttl` | 建立 class / individual 與 object property，讓圖上能看出 SprayLineSystem、Station、Component、SensorMetric、SensorThreshold、State、Cause、ResponseAction、InferenceRule 的關係。 |

## 舉例：檔案裡實際改了什麼

| 檔案 | 具體例子 |
|---|---|
| ``sprayline_full_ontology.ttl`` | 包含 `SprayLineSystem`、`Station`、`Component`、`SensorMetric`、`SensorThreshold`、`State`、`Cause`、`ResponseAction`、`InferenceRule` 等 class。 |
| ``sprayline_full_ontology.ttl`` | 建立 `SprayLine_A -> Station -> Component -> SensorMetric -> SensorThreshold -> State / Cause / ResponseAction` 的展示關係。 |
| ``ontology/0617_B_modular_reference/`` | 保留 0617_B 原本的多檔 ontology，不讓最後版只剩新補的 threshold TTL。 |

## 範例片段

```text
Protégé 展示用：
ontology/sprayline_full_ontology.ttl

Rule 程式推論用：
ontology/sprayline_threshold.ttl
```

## 這樣如何達成任務

我把 ontology 分成兩種：`sprayline_threshold.ttl` 是程式推論用，`sprayline_full_ontology.ttl` 是 Protégé 展示用，展示版補上產線、站點、元件、指標、門檻、狀態、原因與處置建議的完整關係。

## 驗證方式

Protégé 開啟：

```text
ontology/sprayline_full_ontology.ttl
```

看 Classes：

```text
SprayLineSystem
Station
Component
SensorMetric
SensorThreshold
State
Cause
ResponseAction
InferenceRule
```

OntoGraf 搜尋：

```text
SprayLine_A
AirCompressor
Threshold_air_pressure_bar
Rule_Threshold_State_Inference
```


# 8. 修正 Quality、空壓機、噴幅等 UI/API 實測問題

## 主旨

實測時 Quality 沒膜厚資料、空壓機 3.2 被誤判、Station_2 / Station_3 噴幅被誤判，會讓 UI 展示失真。

## 新增了什麼檔案或資料夾

| 新增檔案 / 資料夾 | 用途 |
|---|---|
| `database/patch_sensor_1hour.sql` | 新增 sensor_1hour 補表 SQL，避免 generate_data.py 需要 sensor_1hour 時失敗。 |

## 修改了什麼檔案內容

| 修改檔案 / 資料夾 | 具體修改內容 |
|---|---|
| `database/generate_data.py` | 在 `gen_sensor_reading()` 中加入 `film_thickness_um`、`air_pressure_bar`、`spray_width_mm` 等欄位；寫入 sensor_1min 時也加入 film thickness。 |
| `dataprocess/dataprocess.py` | 在即時資料流程中補 `film_thickness_um` 計算，依 paint_flow、air_pressure、spray_width_error 模擬膜厚。 |
| `api/shaoyu_ui_bridge.py` | 把 quality_module 對應到 `film_thickness_um`，air_compressor 對應 `air_pressure_bar`，spray_width 對應 `spray_width_mm`。 |
| `ui/engineer/app/services/api_data_service.py` | 前端 trend_points 若 metric 是 `film_thickness_um` 且沒有值，會 fallback 到 `quality_score_pct` 或 `quality_score`，避免 Quality 趨勢圖空掉。 |
| `rules/sensor_thresholds.json` | 把 `air_pressure_bar` normal 改為 2.7–3.8，`spray_width_mm` normal 改為 70–130，新增 / 保留 `film_thickness_um` 門檻。 |
| `ui/engineer/config/sensor_thresholds.json` | 同步 UI threshold，避免前後端判斷不一致。 |

## 舉例：檔案裡實際改了什麼

| 檔案 | 具體例子 |
|---|---|
| ``dataprocess/dataprocess.py`` | 膜厚不是固定數字，而是用 `15.0 + (paint_flow - 115.0) * 0.015 + (air_pressure - 3.2) * 0.30 - spray_width_error * 0.015 + noise` 生成，再 clip 到 13.0–17.0。 |
| ``rules/sensor_thresholds.json`` | `air_pressure_bar` normal 明確是 `min: 2.7, max: 3.8`，所以 3.2 不會再誤判。 |
| ``rules/sensor_thresholds.json`` | `spray_width_mm` normal 明確是 `min: 70.0, max: 130.0`，所以 Station_2 約 100、Station_3 約 82 不會誤判。 |
| ``database/generate_data.py`` | 寫入 sensor_1min 時欄位包含 `film_thickness_um`、`air_pressure_bar`、`spray_width_mm`，因此 DB 查詢會有膜厚資料。 |

## 範例片段

```python
# dataprocess/dataprocess.py 舉例
film_thickness = 15.0
film_thickness += (paint_flow - 115.0) * 0.015
film_thickness += (air_pressure - 3.2) * 0.30
film_thickness -= spray_width_error * 0.015
film_thickness = round(float(np.clip(film_thickness, 13.0, 17.0)), 2)
```

```json
// rules/sensor_thresholds.json 舉例
"air_pressure_bar": {
  "normal": { "min": 2.7, "max": 3.8 }
},
"spray_width_mm": {
  "normal": { "min": 70.0, "max": 130.0 }
}
```

## 這樣如何達成任務

我不是只說 Quality 修好了，而是補了資料生成、dataprocess 膜厚計算、API / UI 欄位 mapping、前後端 threshold，讓 Quality、空壓機、噴幅三個實測問題都能在資料源、API、UI 三層對齊。

## 驗證方式

DBeaver：

```sql
SELECT station_id,
       COUNT(film_thickness_um) AS quality_count,
       MIN(film_thickness_um) AS min_film,
       MAX(film_thickness_um) AS max_film
FROM sensor_1min
GROUP BY station_id
ORDER BY station_id;
```

```sql
SELECT station_id, air_pressure_bar, spray_width_mm, film_thickness_um
FROM sensor_1min
ORDER BY ts DESC
LIMIT 20;
```

預期：

```text
film_thickness_um 有值
air_pressure_bar 約 3.0–3.5
spray_width_mm 約 70–130
```


# 9. 補齊 DB catalog / response / cause mapping，避免寫回失敗

## 主旨

Future / Monitoring / Rule 推論出 cause_id 和 response_ids 後，如果 DB 沒有 cause_catalog / response_catalog 對應，alert link 或 future result 寫回會失敗。

## 新增了什麼檔案或資料夾

| 新增檔案 / 資料夾 | 用途 |
|---|---|
| `knowledge/cause_catalog_reference.csv` | 新增 / 保留 0617_B cause catalog 來源參考。 |
| `knowledge/response_catalog_reference.csv` | 新增 / 保留 0617_B response catalog 來源參考。 |
| `database/patch_sensor_1hour.sql` | 新增 sensor_1hour table patch。 |

## 修改了什麼檔案內容

| 修改檔案 / 資料夾 | 具體修改內容 |
|---|---|
| `database/setup_db.sql` | 補 `cause_catalog`、`response_catalog`、`cause_response_map` 的 seed，以及 `sensor_threshold` 補值。 |
| `database/setup_db.sql` | 新增 `sensor_1hour` 建表段落與 index。 |
| `database/generate_data.py` | 配合 DB schema 產生 alert_event、alert_cause_link、alert_response_link、batch_station_status、sensor_1hour 等資料。 |

## 舉例：檔案裡實際改了什麼

| 檔案 | 具體例子 |
|---|---|
| ``database/setup_db.sql`` | 補 cause：`FLOW_IMBALANCE`、`NOZZLE_ANGLE_DRIFT`、`SPRAY_WIDTH_DEVIATION`、`SPRAY_WIDTH_UNSTABLE`、`FILM_THICKNESS_OOC`、`FILM_THICKNESS_VARIATION`。 |
| ``database/setup_db.sql`` | 補 response：`CHECK_FILTER`、`CHECK_SERVO`、`INSPECT_NOZZLE`、`REDUCE_SPEED`、`ADJUST_FILM_THICKNESS`。 |
| ``database/setup_db.sql`` | 補 `cause_response_map`，例如 `FILM_THICKNESS_OOC -> ADJUST_FILM_THICKNESS`、`FLOW_IMBALANCE -> CHECK_FILTER`。 |
| ``database/setup_db.sql`` | 補 `sensor_1hour`，避免 generate_data.py 出現 `relation sensor_1hour does not exist`。 |

## 範例片段

```sql
-- database/setup_db.sql 舉例
INSERT INTO cause_catalog (cause_id, category, description_zh, typical_component, severity) VALUES
  ('SPRAY_WIDTH_DEVIATION', 'quality_fluid', '噴幅偏離各站基準值，可能影響覆蓋範圍', 'spray_width', 'medium'),
  ('FILM_THICKNESS_OOC', 'quality_fluid', '膜厚超出規格範圍', 'quality', 'high')
ON CONFLICT (cause_id) DO UPDATE SET ...

INSERT INTO response_catalog (response_id, description_zh, downtime_estimate_min, skill_required) VALUES
  ('CHECK_FILTER', '檢查濾網壓差與管路流量，必要時更換濾網', 10, 'operator'),
  ('ADJUST_FILM_THICKNESS', '調整膜厚參數，包含速度、流量與空壓比例', 10, 'operator')
ON CONFLICT (response_id) DO UPDATE SET ...
```

## 這樣如何達成任務

我補的是 DB 真的會用到的 catalog 與 mapping，不只是文字說明。這樣 rule 或 monitoring 產生 cause_id / response_ids 後，寫回 alert link 和 future result 時有 DB 對應，不會因缺 catalog 失敗。

## 驗證方式

DBeaver：

```sql
SELECT cause_id, description_zh FROM cause_catalog ORDER BY cause_id;
SELECT response_id, description_zh FROM response_catalog ORDER BY response_id;
SELECT cause_id, response_id FROM cause_response_map ORDER BY cause_id, priority;
```

確認沒有缺：

```text
SPRAY_WIDTH_DEVIATION
FILM_THICKNESS_OOC
CHECK_FILTER
ADJUST_FILM_THICKNESS
```


# 10. 保留 0617_B 少榆端核心資料，包括 CSV、knowledge、schema、service reference、future prediction contract

## 主旨

最後版本不能只剩 final_version；也不能把所有歷史版本亂塞。需要保留 0617_B 中和少榆端服務、future prediction、DB contract、knowledge、rule 有關的核心資料。

## 新增了什麼檔案或資料夾

| 新增檔案 / 資料夾 | 用途 |
|---|---|
| `csv_templates/` | 新增整理 0617_B 核心 CSV templates；四個原本看起來空白的 template 已補 demo row。 |
| `knowledge/` | 新增整理 0617_B cause / response / troubleshooting matrix / legacy thresholds。 |
| `schema/` | 新增整理 0617_B API / DB / service schema。 |
| `templates/` | 新增整理 0617_B JSON payload templates。 |
| `docs/0617_B_reference/` | 新增整理 0617_B contracts、notes、source reference、validation、舊報告 ipynb。 |
| `service_reference_0617_B/` | 新增整理 0617_B future_service、integrated_service、monitoring_worker、troubleshooting_service、time_series_api_src。 |
| `database_versionB_reference_0617_B/` | 新增整理 0617_B Database/versionB reference。 |
| `ontology/0617_B_modular_reference/` | 新增整理 0617_B modular ontology reference。 |

## 修改了什麼檔案內容

| 修改檔案 / 資料夾 | 具體修改內容 |
|---|---|
| `csv_templates/*.csv` | 改成 CP950 / Big5 相容，加入 `sep=,`，讓 Excel 直接雙擊不亂碼並自動分欄。 |
| `csv_templates/alert_event_template.csv` | 補 demo row，包含 event_id、batch_id、station_id、sensor_name、measured_value、state、cause、message。 |
| `csv_templates/batch_run_template.csv` | 補 demo row，包含 batch_id、start_time、ended_time、status。 |
| `csv_templates/sensor_1min_template.csv` | 補 demo row，包含 film_thickness_um、paint_flow_ml_min、filter_diff_pressure_bar、air_pressure_bar、spray_width_mm 等欄位。 |
| `csv_templates/sensor_3min_template.csv` | 補 demo row，包含 gearbox_temperature_c、temperature_c、humidity_rh。 |
| `README_0623ver_2_最終整合說明.md` | 新增說明：哪些是 runtime、哪些是 0617_B reference。 |
| `docs/0623ver_2_核心CSV檔案位置與用途.md` | 新增核心 CSV 對照表，避免老師問 CSV 在哪時找不到。 |

## 舉例：檔案裡實際改了什麼

| 檔案 | 具體例子 |
|---|---|
| ``csv_templates/sensor_1min_template.csv`` | 不再只有欄位，已加入一列 demo：Station_1、film_thickness_um=14.8、air_pressure_bar=3.2、spray_width_mm=99.0 等。 |
| ``service_reference_0617_B/integrated_service/`` | 保留 0617_B 原本 integrated service source，讓老師能追溯少榆端前一版整合服務。 |
| ``docs/0617_B_reference/contracts/future_prediction_result_database_alignment_0616.csv`` | 保留 future prediction result 與 DB 欄位對齊表。 |
| ``knowledge/troubleshooting_matrix_reference.csv`` | 保留 troubleshooting knowledge reference；`config/knowledge/troubleshooting_matrix_reference.csv` 則是 runtime 會讀的位置。 |

## 範例片段

```text
Runtime 會用：
services/
api/
database/
config/knowledge/troubleshooting_matrix_reference.csv

0617_B 追溯用：
service_reference_0617_B/
docs/0617_B_reference/contracts/
csv_templates/
knowledge/
schema/
templates/
```

## 這樣如何達成任務

我保留 0617_B 核心資料，但不是亂塞所有歷史檔。實際會跑的是根目錄 runtime，0617_B 的 CSV、knowledge、schema、contract、service source 則整理到 reference 資料夾供追溯與報告。

## 驗證方式

PowerShell：

```powershell
dir csv_templates
dir knowledge
dir schema
dir templates
dir docs\0617_B_reference
dir service_reference_0617_B
dir database_versionB_reference_0617_B
dir ontology\0617_B_modular_reference
```

Excel 直接開：

```text
csv_templates/sensor_1min_template.csv
csv_templates/alert_event_template.csv
knowledge/cause_catalog_reference.csv
knowledge/response_catalog_reference.csv
```

預期：

```text
中文不亂碼
欄位有分欄
template 有 demo row
```


# 11. 整體驗證清單

完成後建議照這個順序驗證。

## 1. 啟動 Docker 與 API / UI / DB

```powershell
cd "C:\Users\jed92\Desktop\Project-SprayLine\0623ver_2"
powershell -ExecutionPolicy Bypass -File .\start.ps1 -Mode start -WithData
docker compose ps
```

## 2. 開網址

```text
API Swagger: http://localhost:8011/docs
Engineer UI: http://localhost:8013
Manager UI: http://localhost:8012
```

## 3. DBeaver 連線

```text
Host: localhost
Port: 5433
Database: sprayline
Username: postgres
Password: postgres
```

## 4. 測 Quality 資料

```sql
SELECT station_id,
       COUNT(film_thickness_um) AS quality_count,
       MIN(film_thickness_um) AS min_film,
       MAX(film_thickness_um) AS max_film
FROM sensor_1min
GROUP BY station_id
ORDER BY station_id;
```

## 5. 測 Ontology Rule

```powershell
python ontology\threshold_to_ttl.py --csv ontology\threshold_reference.csv --ttl ontology\sprayline_threshold.ttl
python ontology\rule_inference.py --run-smoke-test
```

## 6. Protégé 展示

```text
ontology/sprayline_full_ontology.ttl
```

## 7. Sequence Diagram

```text
docs/sequence_diagram.md
```


In [ ]:
# 可選：在 0623ver_2 根目錄執行，用來檢查本報告提到的重要檔案是否存在。
from pathlib import Path

important_paths = [
    "api/api_server.py",
    "api/service_orchestration_adapter.py",
    "api/shaoyu_ui_bridge.py",
    "api/time_series_service.py",
    "api/d_integration_adapter.py",
    "database/setup_db.sql",
    "database/generate_data.py",
    "database/patch_sensor_1hour.sql",
    "dataprocess/dataprocess.py",
    "rules/sensor_thresholds.json",
    "ui/engineer/config/sensor_thresholds.json",
    "ui/engineer/app/services/api_data_service.py",
    "ontology/threshold_reference.csv",
    "ontology/threshold_to_ttl.py",
    "ontology/sprayline_threshold.ttl",
    "ontology/rule_inference.py",
    "ontology/sprayline_full_ontology.ttl",
    "ontology/0617_B_modular_reference",
    "docs/sequence_diagram.md",
    "docs/0623ver_2_核心CSV檔案位置與用途.md",
    "docs/0623ver_2_Runtime與Reference分工.md",
    "csv_templates/sensor_1min_template.csv",
    "csv_templates/sensor_3min_template.csv",
    "csv_templates/alert_event_template.csv",
    "csv_templates/batch_run_template.csv",
    "knowledge/cause_catalog_reference.csv",
    "knowledge/response_catalog_reference.csv",
    "knowledge/troubleshooting_matrix_reference.csv",
    "config/knowledge/troubleshooting_matrix_reference.csv",
    "docs/0617_B_reference/contracts/future_prediction_result_database_alignment_0616.csv",
    "service_reference_0617_B/integrated_service/sprayline_integrated_service.py",
    "database_versionB_reference_0617_B",
]

root = Path(".")
for rel in important_paths:
    print(f"{'OK' if (root / rel).exists() else 'MISSING'}  {rel}")


# 12. 最後報告總結說法

可以在報告最後這樣收束：

> 這版 `0623ver_2` 不只是整理檔案，而是把老師逐字稿要求的十項工作逐一落到檔案。  
> API 端新增 UI 查詢與 service orchestration endpoint；Bridge 端補 component alias 與 Quality / 空壓機 / 噴幅 mapping；DB 端補 future、alert、catalog、sensor_1hour；資料處理端補 film_thickness_um；rule 端新增 CSV → TTL → Rule pipeline；ontology 端補 Protégé 展示用 full ontology；文件端補 Sequence Diagram。  
> 同時我也把 0617_B 少榆端核心 CSV、knowledge、schema、service source、future prediction contract 乾淨保留下來，讓最後版本既能跑，又能追溯。
